In [5]:
from pathlib import Path
from datetime import datetime
import warnings
import numpy as np
import pandas as pd
import requests
import statsmodels.api as sm
import plotly.graph_objects as go

# ======================== CONFIGURATION ========================
COUNTRY_CODE = "IND"
COUNTRY_NAME = "India"
INDICATOR = "NY.GDP.MKTP.KN"  # GDP (constant local currency units)
START_YEAR = None             # None = earliest available
END_YEAR = None               # None = latest available
HP_LAMBDA = 6.25              # common annual-frequency setting
OUTPUT_DIR = Path("/Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/Teaching-aids")
OUTPUT_HTML_FILE = "animated-potential-GDP.html"
OUTPUT_CSV_FILE = "India_Real_GDP_HP_Kalman_Data.csv"
SAVE_CSV = True
OUTPUT_HTML = OUTPUT_DIR / OUTPUT_HTML_FILE
OUTPUT_CSV = OUTPUT_DIR / OUTPUT_CSV_FILE
API_URL = f"https://api.worldbank.org/v2/country/{COUNTRY_CODE}/indicator/{INDICATOR}"

# ======================== FETCH WORLD BANK DATA ========================
def fetch_world_bank_gdp():
    response = requests.get(API_URL, params={"format": "json", "per_page": 20000}, timeout=60)
    response.raise_for_status()
    payload = response.json()
    if not isinstance(payload, list) or len(payload) < 2 or not payload[1]:
        raise RuntimeError(f"Unexpected or empty World Bank API response: {payload}")

    records = []
    for obs in payload[1]:
        if obs.get("value") is None or obs.get("date") is None:
            continue
        records.append({
            "Year": int(obs["date"]),
            "Real_GDP": float(obs["value"]),
            "Country": obs.get("country", {}).get("value", COUNTRY_NAME),
            "Country_Code": obs.get("countryiso3code", COUNTRY_CODE),
            "Indicator": obs.get("indicator", {}).get("value", "GDP (constant LCU)"),
            "Indicator_Code": obs.get("indicator", {}).get("id", INDICATOR),
        })
    df = pd.DataFrame(records).sort_values("Year").drop_duplicates("Year").reset_index(drop=True)
    if START_YEAR is not None:
        df = df[df["Year"] >= START_YEAR]
    if END_YEAR is not None:
        df = df[df["Year"] <= END_YEAR]
    df = df.reset_index(drop=True)
    if len(df) < 10:
        raise ValueError("At least 10 annual observations are recommended. Widen the sample period.")
    expected = set(range(int(df.Year.min()), int(df.Year.max()) + 1))
    missing = sorted(expected - set(df.Year.astype(int)))
    if missing:
        raise ValueError(f"Missing years within the selected sample: {missing}. Inspect the source; no interpolation was applied.")
    return df


gdp_df = fetch_world_bank_gdp()
gdp_df["Date"] = pd.to_datetime(gdp_df["Year"].astype(str) + "-01-01")
gdp = pd.Series(gdp_df["Real_GDP"].to_numpy(float), index=gdp_df["Date"], name="Real_GDP")
if (gdp <= 0).any():
    raise ValueError("GDP contains non-positive values; inspect the source series.")

# ======================== HP FILTER ON GDP LEVELS ========================
hp_cycle, hp_trend = sm.tsa.filters.hpfilter(gdp, lamb=HP_LAMBDA)
gdp_df["HP_Trend"] = np.asarray(hp_trend)
gdp_df["HP_Deviation"] = gdp_df["Real_GDP"] - gdp_df["HP_Trend"]
gdp_df["HP_Deviation_Percent"] = 100 * gdp_df["HP_Deviation"] / gdp_df["HP_Trend"]

# ======================== KALMAN LOCAL-LINEAR TREND ========================
# Scale GDP for numerical stability; trend is rescaled to original GDP units.
scale = float(gdp.abs().max())
model = sm.tsa.UnobservedComponents(gdp / scale, level="local linear trend")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    result = model.fit(disp=False, maxiter=1000)
gdp_df["Kalman_Trend"] = np.asarray(result.smoothed_state[0]) * scale
gdp_df["Kalman_Deviation"] = gdp_df["Real_GDP"] - gdp_df["Kalman_Trend"]
gdp_df["Kalman_Deviation_Percent"] = 100 * gdp_df["Kalman_Deviation"] / gdp_df["Kalman_Trend"]
gdp_df["Real_GDP_Growth_Percent"] = gdp_df["Real_GDP"].pct_change() * 100

# ======================== EXPORT NUMERICAL DATA ========================
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_columns = [
    "Country", "Country_Code", "Year", "Real_GDP", "HP_Trend", "HP_Deviation",
    "HP_Deviation_Percent", "Kalman_Trend", "Kalman_Deviation",
    "Kalman_Deviation_Percent", "Real_GDP_Growth_Percent"
]
output_df = gdp_df[output_columns].copy()
if SAVE_CSV:
    output_df.to_csv(OUTPUT_CSV, index=False, float_format="%.10f")

# ======================== SHARED PLOT CONFIG ========================
plot_config = {
    "responsive": True,
    "displaylogo": False,
    "displayModeBar": True,
    "scrollZoom": True,
    "modeBarButtonsToAdd": ["zoomIn2d", "zoomOut2d"],
    "toImageButtonOptions": {"format": "png", "filename": "India_Real_GDP_Chart", "scale": 2},
}
dates = gdp_df["Date"]

# Figure 1: GDP level and trend lines (independent toolbar)
fig_gdp = go.Figure()
fig_gdp.add_trace(go.Scatter(x=dates, y=gdp_df["Real_GDP"], mode="lines", name="Real GDP",
    line=dict(color="#173F73", width=3), hovertemplate="Year: %{x|%Y}<br>Real GDP: %{y:,.2f}<extra></extra>"))
fig_gdp.add_trace(go.Scatter(x=dates, y=gdp_df["HP_Trend"], mode="lines", name="HP trend",
    line=dict(color="#D95B55", width=2.5), hovertemplate="Year: %{x|%Y}<br>HP trend: %{y:,.2f}<extra></extra>"))
fig_gdp.add_trace(go.Scatter(x=dates, y=gdp_df["Kalman_Trend"], mode="lines", name="Kalman trend",
    line=dict(color="#31936B", width=2.5), hovertemplate="Year: %{x|%Y}<br>Kalman trend: %{y:,.2f}<extra></extra>"))
fig_gdp.update_layout(title="Real GDP and Estimated Trend", template="plotly_white", autosize=True, height=470,
    margin=dict(l=70, r=25, t=75, b=60), hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0), paper_bgcolor="white", plot_bgcolor="white")
fig_gdp.update_xaxes(title="Year", tickformat="%Y", automargin=True)
fig_gdp.update_yaxes(title="GDP (constant local currency units)", tickformat=",.0f", automargin=True)

# Figure 2: absolute cyclical component (GDP minus trend; independent toolbar)
fig_cycle = go.Figure()
fig_cycle.add_trace(go.Scatter(x=dates, y=gdp_df["HP_Deviation"], mode="lines", name="GDP − HP trend",
    line=dict(color="#D95B55", width=2.3), hovertemplate="Year: %{x|%Y}<br>GDP − HP trend: %{y:,.2f}<extra></extra>"))
fig_cycle.add_trace(go.Scatter(x=dates, y=gdp_df["Kalman_Deviation"], mode="lines", name="GDP − Kalman trend",
    line=dict(color="#31936B", width=2.3), hovertemplate="Year: %{x|%Y}<br>GDP − Kalman trend: %{y:,.2f}<extra></extra>"))
fig_cycle.add_hline(y=0, line_dash="dash", line_color="#64748B", line_width=1.2)
fig_cycle.update_layout(title="Cyclical Component: Real GDP − Trend", template="plotly_white", autosize=True, height=420,
    margin=dict(l=70, r=25, t=75, b=60), hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0), paper_bgcolor="white", plot_bgcolor="white")
fig_cycle.update_xaxes(title="Year", tickformat="%Y", automargin=True)
fig_cycle.update_yaxes(title="Deviation (constant local currency units)", tickformat=",.0f", automargin=True)

# Figure 3: annual real GDP growth bars (independent toolbar)
growth = gdp_df["Real_GDP_Growth_Percent"]
bar_colors = ["#31936B" if pd.notna(v) and v >= 0 else "#D95B55" for v in growth]
fig_growth = go.Figure()
fig_growth.add_trace(go.Bar(x=dates, y=growth, name="Annual real GDP growth", marker_color=bar_colors,
    hovertemplate="Year: %{x|%Y}<br>Growth: %{y:.2f}%<extra></extra>"))
fig_growth.add_hline(y=0, line_color="#64748B", line_width=1)
fig_growth.update_layout(title="Annual Real GDP Growth", template="plotly_white", autosize=True, height=360,
    margin=dict(l=70, r=25, t=75, b=60), showlegend=False, paper_bgcolor="white", plot_bgcolor="white")
fig_growth.update_xaxes(title="Year", tickformat="%Y", automargin=True)
fig_growth.update_yaxes(title="Annual growth (%)", ticksuffix="%", automargin=True)

# ======================== RAW DATA TABLE ========================
table_df = gdp_df[["Year", "Real_GDP", "HP_Trend", "Kalman_Trend", "HP_Deviation", "Kalman_Deviation", "Real_GDP_Growth_Percent"]].copy()
table_df.columns = ["Year", "Real GDP (constant LCU)", "HP trend", "Kalman trend", "GDP − HP trend", "GDP − Kalman trend", "Annual GDP growth (%)"]
for col in table_df.columns[1:]:
    table_df[col] = table_df[col].map(lambda v: f"{v:,.2f}" if pd.notna(v) else "—")
table_html = table_df.to_html(index=False, border=0, classes="data-table", escape=True)

# Embed Plotly.js once; the other two independent figures reuse that embedded library.
plot_gdp_html = fig_gdp.to_html(full_html=False, include_plotlyjs=True, div_id="gdp-level-chart", config=plot_config)
plot_cycle_html = fig_cycle.to_html(full_html=False, include_plotlyjs=False, div_id="gdp-cycle-chart", config=plot_config)
plot_growth_html = fig_growth.to_html(full_html=False, include_plotlyjs=False, div_id="gdp-growth-chart", config=plot_config)
retrieved_at = datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %Z")

# ======================== STANDALONE RESPONSIVE HTML ========================
html_document = f'''<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>{COUNTRY_NAME} Real GDP Trend Dashboard</title>
<style>
:root {{ --ink:#24352B; --muted:#596B60; --line:#C8D5CC; --paper:#F6F8F5; --panel:#FFFFFF; }}
* {{ box-sizing:border-box; }} html,body {{ width:100%; max-width:100%; margin:0; }}
body {{ background:var(--paper); color:var(--ink); font:14px/1.5 Inter,ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif; }}
header {{ background:linear-gradient(120deg,#FBE2C5 0%,#E5F0D8 100%); color:#24352B; padding:22px clamp(14px,3vw,36px); border-bottom:2px solid #C9D8C2; }}
header h1 {{ margin:0 0 6px; font-size:clamp(22px,2.3vw,32px); color:#294735; }} header p {{ margin:4px 0; color:#43594A; }}
main {{ width:100%; max-width:1600px; margin:0 auto; padding:18px clamp(8px,1.5vw,22px) 30px; }}
.controls,.chart-panel,.data-panel {{ background:var(--panel); border:1.5px solid #CBD5CE; border-radius:12px; box-shadow:0 3px 12px rgba(30,55,40,.06); }}
.controls {{ padding:12px 16px; margin-bottom:14px; display:flex; align-items:center; gap:18px; flex-wrap:wrap; }}
.controls label {{ display:inline-flex; align-items:center; gap:7px; font-weight:650; }} .controls input {{ width:17px; height:17px; accent-color:#527D5F; }} .note {{ color:var(--muted); font-size:12px; }}
.chart-panel {{ width:100%; min-width:0; overflow:hidden; padding:8px; margin-bottom:16px; }}
.plot-holder,.js-plotly-plot,.plot-container,.svg-container {{ width:100% !important; max-width:100% !important; }}
.data-panel {{ margin-top:18px; padding:16px; }} .data-panel h2 {{ margin:0 0 8px; font-size:19px; }}
.table-scroll {{ width:100%; overflow:auto; max-height:520px; }} .data-table {{ border-collapse:collapse; width:100%; font-size:12px; white-space:nowrap; }}
.data-table th,.data-table td {{ padding:8px 10px; border-bottom:1px solid #E5ECE6; text-align:right; }}
.data-table th {{ position:sticky; top:0; background:#EDF4EC; color:#2C4734; }} .data-table td:first-child,.data-table th:first-child {{ text-align:left; }}
footer {{ margin-top:16px; color:var(--muted); font-size:12px; }}
@media(max-width:700px) {{ main {{ padding:8px; }} header {{ padding:18px 14px; }} .chart-panel {{ padding:3px; }} .controls {{ gap:10px; }} }}
</style></head><body>
<header><h1>{COUNTRY_NAME}: Real GDP, Potential GDP and Cyclical Component</h1>
<p>World Bank indicator: GDP (constant local currency units) — {INDICATOR}</p>
<p>Period: {int(gdp_df['Year'].min())}–{int(gdp_df['Year'].max())} | Observations: {len(gdp_df)} | HP lambda: {HP_LAMBDA}</p>
<p>Data retrieved: {retrieved_at}</p></header>
<main>
<section class="controls"><strong>Trend visibility:</strong>
<label><input type="checkbox" id="showHP" checked> HP trend and HP deviation</label>
<label><input type="checkbox" id="showKalman" checked> Kalman trend and Kalman deviation</label>
<span class="note">Each figure has its own independent zoom, pan, reset and image-export toolbar.</span></section>
<section class="chart-panel"><div class="plot-holder">{plot_gdp_html}</div></section>
<section class="chart-panel"><div class="plot-holder">{plot_cycle_html}</div></section>
<section class="chart-panel"><div class="plot-holder">{plot_growth_html}</div></section>
<section class="data-panel"><h2>Underlying annual observations</h2>
<p class="note">GDP levels are in constant local currency units. The table includes fitted trends, absolute deviations and annual growth; the CSV retains numeric values.</p>
<div class="table-scroll">{table_html}</div></section>
<footer>Source: World Bank World Development Indicators API. Indicator {INDICATOR}: GDP (constant local currency units). The constant-price reference base may be revised by the source. HP trend uses lambda = {HP_LAMBDA}. Kalman trend uses a local-linear-trend state-space model. These are statistical trend estimates, not official measures of potential GDP.</footer>
</main>
<script>
(function() {{
  const hpBox=document.getElementById("showHP"), kalmanBox=document.getElementById("showKalman");
  function updateVisibility() {{
    Plotly.restyle("gdp-level-chart", {{visible:hpBox.checked}}, [1]);
    Plotly.restyle("gdp-level-chart", {{visible:kalmanBox.checked}}, [2]);
    Plotly.restyle("gdp-cycle-chart", {{visible:hpBox.checked}}, [0]);
    Plotly.restyle("gdp-cycle-chart", {{visible:kalmanBox.checked}}, [1]);
  }}
  hpBox.addEventListener("change",updateVisibility); kalmanBox.addEventListener("change",updateVisibility); updateVisibility();
  window.addEventListener("resize",function() {{ ["gdp-level-chart","gdp-cycle-chart","gdp-growth-chart"].forEach(function(id) {{ const el=document.getElementById(id); if(el) Plotly.Plots.resize(el); }}); }});
}})();
</script></body></html>'''

OUTPUT_HTML.write_text(html_document, encoding="utf-8")
print("Dashboard generation complete.")
print(f"HTML dashboard: {OUTPUT_HTML}")
if SAVE_CSV:
    print(f"Underlying data CSV: {OUTPUT_CSV}")
print(f"Sample: {int(gdp_df['Year'].min())}–{int(gdp_df['Year'].max())}; observations: {len(gdp_df)}")
print("Open the HTML file in a modern browser; the HTML embeds Plotly.js and works offline.")


Dashboard generation complete.
HTML dashboard: /Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/Teaching-aids/animated-potential-GDP.html
Underlying data CSV: /Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/Teaching-aids/India_Real_GDP_HP_Kalman_Data.csv
Sample: 1960–2025; observations: 66
Open the HTML file in a modern browser; the HTML embeds Plotly.js and works offline.


/opt/anaconda3/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency YS-JAN will be used.
  self._init_dates(dates, freq)
